In [4]:
#-----------------------------
# Packages Import
#-----------------------------
%matplotlib inline


import numpy as np
from matplotlib import pyplot as plt

# Used to handle picture drawings
import cv2

# Used Handle directories and paths
import os

# Used to create and compute random angles
import random
import math

# Used to create the image labels
import json



In [5]:
#--------------------------
# Constants across the project
#----------------------------

# Mapping cardinal directions to approximate angles (in radians)
# Starting N at -90 degrees (standard Cartesian setup in image space)
DIRECTIONS = {
    "N":  -math.pi/2, "NE": -math.pi/4, "E": 0, "SE": math.pi/4,
    "S":   math.pi/2, "SW": 3*math.pi/4, "W": math.pi, "NW": -3*math.pi/4
}

DIRECTIONS_NUMBER= {
    "N":  1, "NE": 2, "E": 3, "SE": 4,
    "S":   5, "SW": 6, "W": 7, "NW": 8
}

TYPES_NUMBER= {
    "Under":  0, "Over": 1, "None":2, "empty": 3, "full": 4,
}

TYPES_NUMBER_INV= {
    0: "Under",  1:"Over", 2:"None", 3: "empty", 4:"full",
}



DIR_KEYS = list(DIRECTIONS.keys())


In [6]:
#------------------------
#Basic Drawing functions:
#-----------------------

def get_boundary_point(angle, size):
    """Calculates where a line at 'angle' hits the image boundary."""
    h, w = size
    cx, cy = w // 2, h // 2
    
    # Large distance to ensure we hit the boundary
    dist = max(w, h)
    
    x1 = int(cx - dist * math.cos(angle))
    y1 = int(cy - dist * math.sin(angle))
    x2 = int(cx + dist * math.cos(angle))
    y2 = int(cy + dist * math.sin(angle))
    
    _,p1,p2 = cv2.clipLine((0, 0, w, h), (x1, y1), (x2, y2))
    return [p1,p2] 


#This function Draws a line going through the center given the angle
def draw_strand(img, angle, thickness, isMain):
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    
    p_in, p_out= get_boundary_point(angle, (h, w))
    
    cv2.line(img, p_in, p_out, (0,0,0), thickness, cv2.LINE_AA)
    #if isMain: cv2.circle(img,p_in, 15, (0,0,0), -1)
    
    return None

#15 is the circle radius

#This function Draws an overline line going through the center given the angle


def draw_over_strand(img, angle, thickness, space_bewteen_strands, isMain):
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    
    p_in, p_out= get_boundary_point(angle, (h, w))
    
    cv2.line(img, p_in, p_out, (255,255,255), thickness+space_bewteen_strands, cv2.LINE_AA)
    cv2.line(img, p_in, p_out, (0,0,0), thickness, cv2.LINE_AA)
    #if isMain: cv2.circle(img,p_in, 15, (0,0,0), -1)
    return None


In [7]:
#------------------
# Advanced drawing functions
#------------------


#------------------
# Constants
#----------------------
WINDOW_SIZE = (64, 64) 
#THICKNESS = 2
SPACE_BEWTEEN_STRANDS=8

#--------------------------------------------
# Drawing functions
#-----------------------------------------
def draw_Straight_Line(direction, thickness):
    #Randomize the main direction
    angle = DIRECTIONS[DIR_KEYS[direction]]+ random.uniform(-1, 1)*math.pi/16
    draw_strand(img, angle, thickness  * 2, True)
    return None


def draw_over_crossing(direction1, direction2, thickness):

    #Randomize a small tilt of the main picture
    angle1 = DIRECTIONS[DIR_KEYS[(direction1 + 2) % 8]]+ random.uniform(-1, 1)*math.pi/16
    angle2 = DIRECTIONS[DIR_KEYS[(direction2 + 2) % 8]]+ random.uniform(-1, 1)*math.pi/16

    # Draw secondary strand FIRST (it goes underneath)
    draw_strand(img, angle2, thickness  * 2, False)
    # Draw primary strand SECOND (it goes on top)
    draw_over_strand(img, angle1, thickness * 2, SPACE_BEWTEEN_STRANDS, True)

    return None


def draw_under_crossing(direction1, direction2, thickness):
        #Randomize a small tilt of the main picture
    angle1 = DIRECTIONS[DIR_KEYS[(direction1 + 2) % 8]]+ random.uniform(-1, 1)*math.pi/16
    angle2 = DIRECTIONS[DIR_KEYS[(direction2 + 2) % 8]]+ random.uniform(-1, 1)*math.pi/16

    # Draw primary strand FIRST (it goes underneath)
    draw_strand(img, angle1, thickness  * 2, True)
    # Draw secondart strand SECOND (it goes on top)
    draw_over_strand(img, angle2, thickness  * 2, SPACE_BEWTEEN_STRANDS, False)

    return None

In [ ]:
#-------------------------------
# Small cell that generates 5 random corssing
#-------------------------------

N=10
thickness=2

for i in range(N):
    # Initialize a clean, gray background
    img = np.full((WINDOW_SIZE[1], WINDOW_SIZE[0], 3), -1, dtype=np.uint8)
    
    
    # 2. Randomize the crossing type (33% No, 33% Over, 33% Under)
    
    cross_choice = random.random()
    main_direction, other_direction = random.sample(range(1, 7), 2)

    is_thin= random.randrange(5)
    #0- too thin empy image,  1-2-3 normal image, 4-thich image almost all
    
    if is_thin ==0:
        crossing_type= "empty"
        print("thin")
    elif is_thin== 4:
        crossing_type= "full"
        draw_Straight_Line(main_direction, 20)
        print("full")
    else:    
        if cross_choice < 0.33:
            # State: NONE
            crossing_type = "None"
            draw_Straight_Line(main_direction, thickness) 

        elif cross_choice >= 0.33 and cross_choice < 0.66:
            # State: OVER-CROSSING
            crossing_type = "Over"
            draw_over_crossing(main_direction, other_direction, thickness)  

        else:
            # State: UNDER-CROSSING
            crossing_type = "Under"
            draw_under_crossing(main_direction, other_direction, thickness)

        
    #Invert colors, this might be good for recognition
    inv_img = cv2.bitwise_not(img)
    gray_img = cv2.cvtColor(inv_img, cv2.COLOR_BGR2GRAY)
    normalized_img=  gray_img/255.0
    
  
    
    
    plt.imshow(normalized_img)
    plt.show()
    

In [ ]:
#-------------------------------------
# Image creation loop
#--------------------------------------


OUTPUT_DIR = "local_Knot_training_data"
NUM_IMAGES = 1000 # Start with this to test



# current working directory
path = os.getcwd()
# data directory
DATA_DIR = os.path.join(path, os.pardir,'data',OUTPUT_DIR)

# Ensure output directory exists
os.makedirs( DATA_DIR, exist_ok=True)
#os.makedirs(os.path.join(data_, "images"), exist_ok=True)

dataset_labels = []

for i in range(NUM_IMAGES):
    # Initialize a clean, gray background
    img = np.full((WINDOW_SIZE[1], WINDOW_SIZE[0], 3), -1, dtype=np.uint8) 

    # 2. Randomize the crossing type (33% No, 33% Over, 33% Under)
    cross_choice = random.random()
    main_direction, other_direction = random.sample(range(1, 7), 2)

    is_thin= random.randrange(5)
    #0- too thin empy image,  1-2-3 normal image, 4-thich image almost all
    
    if is_thin ==0:
        crossing_type= "empty"
    elif is_thin== 4:
        crossing_type= "full"
        draw_Straight_Line(main_direction, 20)
    else:    
        if cross_choice < 0.33:
            # State: NONE
            crossing_type = "None"
            draw_Straight_Line(main_direction, thickness) 

        elif cross_choice >= 0.33 and cross_choice < 0.66:
            # State: OVER-CROSSING
            crossing_type = "Over"
            draw_over_crossing(main_direction, other_direction, thickness)  

        else:
            # State: UNDER-CROSSING
            crossing_type = "Under"
            draw_under_crossing(main_direction, other_direction, thickness)

        
    
    # Invert the image and just consider the intensity
    img = cv2.bitwise_not(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img=img/255.0
    
    
    #Save Image and Metadata
    img_filename = f"knot_{i:05d}.jpg"
    img_path = os.path.join(DATA_DIR, img_filename)
    cv2.imwrite(img_path, img)
    
    dataset_labels.append({
        "filename": img_filename,
        "label_index": i,
        "direction": DIR_KEYS[main_direction],
        "crossing": crossing_type,
    })
    

# Save labels to JSON
with open(os.path.join(DATA_DIR, "labels.json"), "w") as f:
    json.dump(dataset_labels, f, indent=4)

print(f"Generated {NUM_IMAGES} synthetic images in '{OUTPUT_DIR}'")

In [1]:
'''General comments about the image creation. So it created only straigh lines, although close enough all lineas are roughly
straigt, it would be nice to add a functionality of the code that randimly twist and curves the generated lines.'''

In [ ]:
import cv2
import numpy as np
import torch

class RotationalTwist(object):
    def __init__(self, max_twist=0.05):
        self.max_twist = max_twist # How much "spiral" to add

    def __call__(self, img):
        # Convert tensor to numpy if needed
        if isinstance(img, torch.Tensor):
            img = img.numpy().transpose(1, 2, 0)

        h, w = img.shape[:2]
        cx, cy = w // 2, h // 2
        
        # Create a coordinate grid
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        
        # Shift origin to center
        x_c = x - cx
        y_c = y - cy
        
        # Calculate radius and angle for every pixel
        r = np.sqrt(x_c**2 + y_c**2)
        theta = np.arctan2(y_c, x_c)
        
        # Apply the twist: angle changes based on radius
        # (Randomize the strength so every image is different)
        strength = np.random.uniform(-self.max_twist, self.max_twist)
        new_theta = theta + strength * r
        
        # Map back to Cartesian coordinates
        map_x = cx + r * np.cos(new_theta)
        map_y = cy + r * np.sin(new_theta)
        
        # Remap the image pixels to the new grid
        twisted = cv2.remap(img.astype(np.float32), 
                           map_x.astype(np.float32), 
                           map_y.astype(np.float32), 
                           cv2.INTER_LINEAR)
        
        # Return as tensor
        if len(twisted.shape) == 2: # Keep 1-channel shape
            twisted = twisted[:, :, np.newaxis]
        return torch.from_numpy(twisted.transpose(2, 0, 1))

In [ ]:
from torchvision.transforms import v2
import torch



input_tranform= v2.Compose([
    # Ensure it's a tensor and floats
    v2.ToImage(),

    # Change to tensor
    v2.ToDtype(torch.float32, scale=True), # This also normalizes 0-255 to 0-1!
   
    # 4. Final resize just in case
    v2.Resize((64, 64), antialias=True),
    
    # 5. Normalization (Optional but helpful)
    v2.Normalize(mean=[0.5], std=[0.5]) # Centers data around 0
])
    
    
    

# Define the augmentation pipeline
train_transforms = v2.Compose([
    # Ensure it's a tensor and floats
    v2.ToImage(),
    # Make twist with OpenCV
    RotationalTwist(max_twist=0.03),
    
    # Change to tensor
    v2.ToDtype(torch.float32, scale=True), # This also normalizes 0-255 to 0-1!
    
    # 2. Geometric Augmentation (The "Jitter")
    v2.RandomRotation(degrees=15),          # Tilt the lines/circles up to 15°
    v2.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)), # Shift and zoom
    
    # 3. Handle Noise (Synthetic data is often too "clean")
    v2.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), 
    
    # 4. Final resize just in case
    v2.Resize((64, 64), antialias=True),
    
    # 5. Normalization (Optional but helpful)
    v2.Normalize(mean=[0.5], std=[0.5]) # Centers data around 0
])


In [202]:
# current working directory
path = os.getcwd()
# data directory
DATA_DIR = os.path.join(path, os.pardir,'data','local_Knot_training_data')

In [203]:
# Open and load the file
with open(os.path.join(DATA_DIR, "labels.json"), "r") as file:
    data = json.load(file)

#Creates an iterable with the filenames
filename=[label['filename'] for label in data ]
directions=[label['direction'] for label in data ]
types=[label['crossing'] for label in data ]



In [204]:
# Dataloader and Data functions
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import datasets

# Pytorch image functions
import torchvision
from torchvision.io import decode_image
from torchvision.io import read_image



In [205]:
#-----------------------------------------------
# This block Loads the data to train the model
#-----------------------------------------------

# I am defining a Dataset in torch following the tutorial at
# https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
class LocalKnotImageDataset(Dataset):
    def __init__(self, annotations_file, directions_file, type_file,img_dir, transform=None, target_transform=None):
        self.annotations_file = annotations_file
        #Ineed to change this to read my json data file. Not just to a given list of file names
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform
        self.directions_file= directions_file
        self.type_file= type_file
        
    def __len__(self):
        return len(self.annotations_file)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.annotations_file[idx])
        #change the way I get  img_path
        #print(img_path)
        #-------------------------
        # TODO: THERE IS AN ISSUE WITH MY TORCH VERSION AND HAVE TO USE READ IMAGE INSTEAD OF DECODE IMAGE
        #---------------------------
        image = read_image(img_path)
        direction = DIRECTIONS_NUMBER[self.directions_file[idx]]
        
        types = TYPES_NUMBER[self.type_file[idx]]
    
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            direction = self.transform(direction)
        return image, types

In [206]:
#----------------------------------------
# We now prepare the train and test data
#----------------------------------------

#I have to change these methods to fit my custom datasets
training_data = LocalKnotImageDataset(
    annotations_file=filename, directions_file= directions, type_file=types,
    img_dir=DATA_DIR, transform=train_transforms
)

test_data = LocalKnotImageDataset(
    annotations_file=filename, directions_file= directions, type_file=types,
    img_dir=DATA_DIR
)


BATCH_SIZE=32



train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

In [207]:
'''
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader


# Apply transforms to training only
train_dataset = ImageFolder(DATA_DIR, transform=train_transforms)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# For validation, use a "clean" transform (only Resize and Normalize)
val_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((64, 64), antialias=True),
    v2.Normalize(mean=[0.5], std=[0.5])
])

'''

'\nfrom torchvision.datasets import ImageFolder\nfrom torch.utils.data import DataLoader\n\n\n# Apply transforms to training only\ntrain_dataset = ImageFolder(DATA_DIR, transform=train_transforms)\ntrain_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)\n\n# For validation, use a "clean" transform (only Resize and Normalize)\nval_transforms = v2.Compose([\n    v2.ToImage(),\n    v2.ToDtype(torch.float32, scale=True),\n    v2.Resize((64, 64), antialias=True),\n    v2.Normalize(mean=[0.5], std=[0.5])\n])\n\n'

In [2]:
#--------------------------------------------------
# Check if the iterables in dataloader are working
#.--------------------------------------------------

from torchvision.transforms.functional import to_pil_image

# Display image and label.
train_features, train_labels = next(iter(train_dataloader))

train_features.size()

print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {len(train_labels)}")

#img = train_features[10].squeeze()
img = train_features[:10]
label = train_labels[:10]


#to_pil_image is necesarry because the tensor is of size [3,64,64] while plt.imshow take forms [64,64,3]
#Also plt takes numpyArrays and not tensors.
for i in range(len(img)):
    print(img[0][0][:][30])
    plt.imshow(to_pil_image(img[i]))
    typ=TYPES_NUMBER_INV[label[i].item()]
    print(typ)
    plt.show()



NameError: name 'train_dataloader' is not defined

In [217]:
'''
-------------------- BIG CHANGES ---------------------------------
Change from an 8 classifier to a two head clasiffier:
- One to detect if a line, over-crossing, undercrossing
- Find the direction of the main line


The input images will have 4 channels:
1. the picture channel (just the intensity),
2. An image with a dot stating where the arrow begins,
3. X-Sobel channel to se x-direction gradients
4. Y-Sobel channel to se y-direction gradients

This will helfully will help the model predict better.
Also the layer of these classification should be different:
The convolution for the finding whether is over-crossing or undercossing should quite fine (so it can detect the line gap)
The convolution to predict the direction and where it start can be bigger 
'''





import torch
import torch.nn as nn


#Hyperparameter Constants 
channel1_in=1
channel1_out=4
channel2_out=16

firstLayer=128
secondLayer=6

'''
5-Possible classifications: No line (underfocus) , to much line (overfocus), line, under-crossing, over-crossing
'''

#----------------------------
# The architecture I used is the one similar to digit classifier 
# 2D-CONVOLUTION -> ReLU -> MAXPOOL -> 2D-CONVOLUTION -> ReLu -> Flatten
#
# This does not work at all. Need to find architerture.
#
#-----------------------------
class Neural_Network(nn.Module):
    def __init__(self):
        super().__init__()
# 1. Use larger kernels (5x5 or 7x7) for the first layer.
        # This helps the model "see" a longer stretch of a line at once.
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, padding=2) 
        
        # 2. BatchNorm is essential! It prevents the 'nan' and 'not learning' issues
        # by keeping the math centered and scaled.
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(2) # 64x64 -> 32x32 -> 16x16
        
        self.fc_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3), # Prevents the model from just 'memorizing' your synthetic lines
            nn.Linear(128, 5),
            nn.Softmax(dim=1) 
        )

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        return self.fc_stack(x)

    '''        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(channel1_in, channel1_out, kernel_size=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(channel1_out, channel2_out, kernel_size=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
            #TODO READ ABOUT ADAPTATIVEAVGPOOL
        )
        self.fc = nn.Sequential(
            nn.Linear( channel2_out*channel2_out, firstLayer),
            #nn.Linear(conv2_out, firstLayer),
            nn.ReLU(),
            nn.Linear(firstLayer, secondLayer),
            nn.Softmax(dim=1) # Probabilities for the 40 states 8 lines + 16 under crossings+ 16 over crossings 
        )

    def forward(self, x):
        #print(f"The shape of the input tensor is: {x.shape}")
        x = self.conv_layers(x)
        #print(f"The shape of the input tensor after convolution is: {x.shape}")
        x = x.view(x.size(0), -1)
        #print(f"The shape of the input tensor after reshaping is: {x.shape}")
        return self.fc(x)
'''
    
    
model = Neural_Network() 

In [218]:
#------------------------------
# This block trains the model
#------------------------------


#-----------------------------------------
# Training loop
#-----------------------------------------
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode 
    # Not necessary for this case, since there is no batch renormalization layer, but is a good practice
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        #As given X is a tensor of integers have to cast them to float
        X=X.float()
        pred = model(X)
        
        #print(f'prediction is {pred} and the true output is {y}')
        
        loss = loss_fn(pred, y)
        
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
       # print(batch)
       # print(len(X))
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode  
    # Not necessary for this case, since there is no batch renormalization layer, but is a good practice
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            #As given X is a tensor of integers have to cast them to float
            X=X.float()
            #print(X.shape)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            #print(model(X))
            #print(f"the predicted output is: {y}")
            #print(f"the model output is:{pred.argmax(1)}")
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    
    


# Hyperparameters constants
learning_rate = 0.5
batch_size = 32
epochs = 100



'''
We will use a Cross Entropy loss function. This is the standard for multi-class image classification. 
It ensures that the model outputs high probabilities for the correct class and low probabilities for others.
'''
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

# We will use a Stochastic Gradiendt descent for the update of the model. This is standard
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.605105  [   32/ 1000]
Test Error: 
 Accuracy: 19.8%, Avg loss: 1.705520 

Epoch 2
-------------------------------
loss: 1.592333  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.674364 

Epoch 3
-------------------------------
loss: 1.373583  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.680223 

Epoch 4
-------------------------------
loss: 1.527428  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.683153 

Epoch 5
-------------------------------
loss: 1.529998  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.680223 

Epoch 6
-------------------------------
loss: 1.561082  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.674364 

Epoch 7
-------------------------------
loss: 1.404833  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.680223 

Epoch 8
-------------------------------
loss: 1.436082  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.674364 

Epoch 9
----------------

Test Error: 
 Accuracy: 22.7%, Avg loss: 1.677294 

Epoch 69
-------------------------------
loss: 1.467333  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.671434 

Epoch 70
-------------------------------
loss: 1.498583  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.677294 

Epoch 71
-------------------------------
loss: 1.529833  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.677294 

Epoch 72
-------------------------------
loss: 1.623583  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.680223 

Epoch 73
-------------------------------
loss: 1.498583  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.677294 

Epoch 74
-------------------------------
loss: 1.373583  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.683153 

Epoch 75
-------------------------------
loss: 1.404833  [   32/ 1000]
Test Error: 
 Accuracy: 22.7%, Avg loss: 1.677294 

Epoch 76
-------------------------------
loss: 1.373583  [   32/ 1000]
Test Error: 
 Ac

In [ ]:
#---------------------------------------------------------------------------------------------
# This gives an interface to see if the model is well trained, testing in hand drawn examples
#---------------------------------------------------------------------------------------------

import tkinter as tk
from PIL import Image, ImageDraw
import numpy as np

class DrawerApp:
    def __init__(self, model_callback):
        self.root = tk.Tk()
        self.root.title("AI Image Classifier")
        self.model_callback = model_callback # Your model function goes here
        
        # Create Canvas
        self.canvas = tk.Canvas(self.root, width=280, height=280, bg='white')
        self.canvas.grid(row=0, column=0, columnspan=2)
        
        # Image buffer (to save what we draw)
        self.image = Image.new("L", (280, 280), "white")
        self.draw = ImageDraw.Draw(self.image)
        
        # Bind Mouse Events
        self.canvas.bind("<B1-Motion>", self.paint)
        
        # Buttons
        tk.Button(self.root, text="Classify", command=self.classify).grid(row=1, column=0)
        tk.Button(self.root, text="Clear", command=self.clear).grid(row=1, column=1)
        
    def paint(self, event):
        x1, y1 = (event.x - 8), (event.y - 8)
        x2, y2 = (event.x + 8), (event.y + 8)
        self.canvas.create_oval(x1, y1, x2, y2, fill="black", width=12)
        self.draw.ellipse([x1, y1, x2, y2], fill="black")

    def clear(self):
        self.canvas.delete("all")
        self.image = Image.new("L", (280, 280), "white")
        self.draw = ImageDraw.Draw(self.image)

    def classify(self):
        # Resize image to match model input 
        img_resized = self.image.resize((64, 64))
        
        # Convert to NumPy and invert colors (often needed if model was trained on black background)
        img_array = np.array(img_resized)
        img_array = np.invert(img_array) 
        
        # Normalize and reshape for your specific model
        img_array = img_array / 255.0 -1
        img_final = img_array.reshape(64, 64) # Example shape
        
        
        # Call your model!
        plt.imshow(img_final)
        plt.show()
        prediction = self.model_callback(img_final)
        
        
        print(f"I think you drew a:{prediction}")

    def run(self):
        self.root.mainloop()

def my_model_predict(data):
    # call model.predict(data)
    data_tensor=input_tranform(data)

    model.eval() # 1. Set model to evaluation mode (turns off dropout/batchnorm)
    
    print(data_tensor.shape)
    
    with torch.no_grad():
    
        data_tensor = data_tensor.unsqueeze(0)
        pred = model(data_tensor)
        
    return pred

app = DrawerApp(my_model_predict)
app.run()


In [ ]:
TYPES_NUMBER= {
    "Under":  1, "Over": 2, "None":3, "empty": 4, "full": 5,
}

TYPES_NUMBER_INV= {
    1: "Under",  2:"Over", 3:"None", 4: "empty", 5:"full",
}
